# Obtain
Obtain the challenge data set and the weather data

In [ ]:
import pandas as pd

df = pd.read_csv("Training_data_uhi_index_2025-02-18.csv")

In [ ]:
display(df)

,Longitude,Latitude,datetime,UHI Index
0,-73.909167,40.813107,24-07-2021 15:53,1.030289
1,-73.909187,40.813045,24-07-2021 15:53,1.030289
2,-73.909215,40.812978,24-07-2021 15:53,1.023798
3,-73.909242,40.812908,24-07-2021 15:53,1.023798
4,-73.909257,40.812845,24-07-2021 15:53,1.021634
...,...,...,...,...
11224,-73.957050,40.790333,24-07-2021 15:57,0.972470
11225,-73.957063,40.790308,24-07-2021 15:57,0.972470
11226,-73.957093,40.790270,24-07-2021 15:57,0.981124
11227,-73.957112,40.790253,24-07-2021 15:59,0.981245


In [ ]:
df.shape

(11229, 4)

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import time
from tqdm import tqdm

# (If using tqdm with pandas.apply, enable it as follows)
tqdm.pandas()

# Assuming df is already loaded and df['datetime'] is a datetime column.
# Extract hour-based timestamps for API queries (using 'h' instead of 'H')
df['hour'] = df['datetime'].dt.floor('h')  # e.g., 15:00
df['next_hour'] = df['hour'] + pd.Timedelta(hours=1)  # e.g., 16:00

# Get unique queries: unique combination of (Latitude, Longitude, hour, next_hour)
unique_queries = df[['Latitude', 'Longitude', 'hour', 'next_hour']].drop_duplicates()

# Dictionary to store API results for each unique (Latitude, Longitude, hour)
temperature_data = {}

def get_hourly_temperatures(latitude, longitude, hour, next_hour):
    # Use only the date portion for the API call
    date_str = hour.strftime('%Y-%m-%d')
    url = (f"https://api.open-meteo.com/v1/forecast?"
           f"latitude={latitude}&longitude={longitude}&hourly=temperature_2m&"
           f"start_date={date_str}&end_date={date_str}&timezone=auto")
    try:
        # Delay added to help prevent rate limiting
        time.sleep(5)
        response = requests.get(url)
        response.raise_for_status()
        data = json.loads(response.text)
        if 'hourly' in data and 'temperature_2m' in data['hourly']:
            times = pd.to_datetime(data['hourly']['time'])
            temps = data['hourly']['temperature_2m']
            return dict(zip(times, temps))
        else:
            print(f"Error: Invalid response from API for lat {latitude}, lon {longitude} on {date_str}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching temperature for {latitude}, {longitude}: {e}")
        return None

def fetch_and_store(row):
    lat, lon, hour, next_hour = row['Latitude'], row['Longitude'], row['hour'], row['next_hour']
    key = (lat, lon, hour)
    if key not in temperature_data:
        temp_data = get_hourly_temperatures(lat, lon, hour, next_hour)
        if temp_data:
            temperature_data[key] = temp_data

# Use multi-threading to speed up API calls with a tqdm progress bar
rows = [row for _, row in unique_queries.iterrows()]
with ThreadPoolExecutor(max_workers=10) as executor:
    list(tqdm(executor.map(fetch_and_store, rows), total=len(rows), desc="Fetching temperature data"))

def interpolate_temperature(row):
    lat, lon, dt, hour, next_hour = row['Latitude'], row['Longitude'], row['datetime'], row['hour'], row['next_hour']
    key = (lat, lon, hour)
    if key in temperature_data:
        hourly_data = temperature_data[key]

        # Try to get the exact hourly temperature values
        temp1 = hourly_data.get(hour)
        temp2 = hourly_data.get(next_hour)

        # If an exact match isn't found, find the closest available time
        if temp1 is None:
            closest = min(hourly_data.keys(), key=lambda t: abs((t - hour).total_seconds()))
            temp1 = hourly_data[closest]
        if temp2 is None:
            closest = min(hourly_data.keys(), key=lambda t: abs((t - next_hour).total_seconds()))
            temp2 = hourly_data[closest]

        # Linear interpolation for minute-level value
        total_minutes = (next_hour - hour).total_seconds() / 60
        elapsed_minutes = (dt - hour).total_seconds() / 60
        interpolated_temp = temp1 + (temp2 - temp1) * (elapsed_minutes / total_minutes)
        return interpolated_temp
    return np.nan

# Apply the interpolation to each row with a tqdm progress bar
df['temperature'] = df.progress_apply(interpolate_temperature, axis=1)

# Save the result
df.to_csv("output_with_interpolated_temperatures.csv", index=False)


Fetching temperature data:   6%|▌         | 662/11229 [05:00<1:51:13,  1.58it/s]

Error fetching temperature for 40.76598333, -73.97190333: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)Error fetching temperature for 40.76605667, -73.97185167: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)

Error fetching temperature for 40.76593333, -73.971965: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.76741333, -73.97092667: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)Error fetching temperature for 40.76717667, -73.97117167: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.76697833, -73.971305: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.76738833, -73.97095833: HTTPSConnectionPool(host='api.open-

Fetching temperature data:   6%|▌         | 663/11229 [21:43<237:01:00, 80.76s/it]

Error fetching temperature for 40.820625, -73.92858: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)Error fetching temperature for 40.82068333, -73.92856833: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)

Error fetching temperature for 40.82038833, -73.92867667: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.77478, -73.98248667: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.77478&longitude=-73.98248667&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.77462167, -73.98253167: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.77462167&longitude=-73.98253167&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=autoError fetching temperature f

Fetching temperature data:  12%|█▏        | 1366/11229 [30:47<4:52:08,  1.78s/it] 

Error fetching temperature for 40.77429833, -73.9572: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)Error fetching temperature for 40.782075, -73.986545: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)

Error fetching temperature for 40.829855, -73.929745: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.829855&longitude=-73.929745&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.831145, -73.93181667: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.831145&longitude=-73.93181667&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.83324333, -73.930995: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.83324333&longitude=-73.930995&hourly

... (148 outputs left out)

Fetching temperature data:  79%|███████▉  | 8889/11229 [3:48:05<1:28:15,  2.26s/it]

Error fetching temperature for 40.80972333, -73.95887167: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.804885, -73.94122: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.804885&longitude=-73.94122&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.79804333, -73.953105: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.79804333&longitude=-73.953105&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.82167167, -73.950285: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.82167167&longitude=-73.950285&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.798045, -73.95313667: 429 Client Error: 

Fetching temperature data:  79%|███████▉  | 8924/11229 [3:49:39<1:30:28,  2.35s/it]

Error fetching temperature for 40.818745, -73.95999333: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.79559667, -73.93424667: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.812345, -73.95691667: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=None)
Error fetching temperature for 40.81781667, -73.95411833: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.81781667&longitude=-73.95411833&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching temperature for 40.79790333, -73.95424667: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/forecast?latitude=40.79790333&longitude=-73.95424667&hourly=temperature_2m&start_date=2021-07-24&end_date=2021-07-24&timezone=auto
Error fetching tempe

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import time

# Ensure your DataFrame 'df' is loaded and has a proper datetime column.
# For example:
# df = pd.read_csv("your_data.csv")
# df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')

# Create a column for the date (normalized to midnight) to use for API queries.
df['date'] = df['datetime'].dt.normalize()  # e.g., 2021-07-24 00:00:00

# Dictionary to store photoperiod results for each unique (Latitude, Longitude, date)
photoperiod_data = {}

def get_photoperiod(latitude, longitude, date):
    """
    Gets photoperiod data using the Sunrise-Sunset API.
    """
    try:
        url = (f"https://api.sunrise-sunset.org/json?"
               f"lat={latitude}&lng={longitude}&date={date.strftime('%Y-%m-%d')}&formatted=0")
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        data = json.loads(response.text)
        if data['status'] == 'OK':
            sunrise = datetime.fromisoformat(data['results']['sunrise'])
            sunset = datetime.fromisoformat(data['results']['sunset'])
            day_length_seconds = (sunset - sunrise).total_seconds()
            return day_length_seconds / 3600  # Convert seconds to hours
        else:
            print(f"Sunrise-Sunset API Error: {data['status']}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching photoperiod: {e}")
        return None

# Get unique queries: unique combination of (Latitude, Longitude, date)
unique_queries_photo = df[['Latitude', 'Longitude', 'date']].drop_duplicates()

def fetch_photoperiod(row):
    lat, lon, date = row['Latitude'], row['Longitude'], row['date']
    key = (lat, lon, date)
    if key not in photoperiod_data:
        # Optional: a small delay to reduce the chance of hitting rate limits
        time.sleep(1)
        photoperiod = get_photoperiod(lat, lon, date)
        photoperiod_data[key] = photoperiod

# Use multi-threading with a tqdm progress bar for fetching photoperiod data
rows = [row for _, row in unique_queries_photo.iterrows()]
with ThreadPoolExecutor(max_workers=10) as executor:
    list(tqdm(executor.map(fetch_photoperiod, rows), total=len(rows), desc="Fetching photoperiod data"))

# Now, map the fetched photoperiod data back to the original DataFrame
def get_photoperiod_from_data(row):
    key = (row['Latitude'], row['Longitude'], row['date'])
    return photoperiod_data.get(key, None)

# Use tqdm with pandas apply to show progress
tqdm.pandas(desc="Mapping photoperiod data")
df['photoperiod'] = df.progress_apply(get_photoperiod_from_data, axis=1)

# Save the updated DataFrame to a CSV file
df.to_csv("output_with_photoperiod.csv", index=False)


Mapping photoperiod data: 100%|██████████| 11229/11229 [00:00<00:00, 54114.24it/s]


# Scrub

In [ ]:
# prompt: order by datetime

import pandas as pd
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values(by='datetime')
display(df)


In [ ]:
# prompt: generate a feature called minute which represents the minute portion of the datetime

df['minute'] = df['datetime'].dt.minute


# Explore

In [ ]:
min(df.datetime), max(df.datetime)

In [ ]:
import pandas as pd
from datetime import datetime

def analyze_datetime_granularity_df(df, datetime_col='datetime'):
    """
    Analyze the granularity of datetime values in a DataFrame

    Parameters:
    df: pandas DataFrame
    datetime_col: name of the datetime column (default='datetime')
    """
    # Ensure datetime column is datetime type
    if not pd.api.types.is_datetime64_any_dtype(df[datetime_col]):
        df[datetime_col] = pd.to_datetime(df[datetime_col])

    # Analyze time differences
    time_diffs = df[datetime_col].diff().dropna()

    analysis = {
        'min_difference': time_diffs.min(),
        'max_difference': time_diffs.max(),
        'unique_times': df[datetime_col].nunique(),
        'total_records': len(df),
        'has_seconds': any('S' in str(diff) for diff in time_diffs),
        'has_milliseconds': any('ms' in str(diff) for diff in time_diffs),
        'time_range': df[datetime_col].max() - df[datetime_col].min(),
        'sample_values': df[datetime_col].head().tolist()
    }

    return analysis

def format_time_analysis(analysis):
    """
    Format the analysis results into a readable string
    """
    return f"""Datetime Granularity Analysis:

- Smallest time difference: {analysis['min_difference']}
- Largest time difference: {analysis['max_difference']}
- Number of unique timestamps: {analysis['unique_times']}
- Total records: {analysis['total_records']}
- Contains seconds: {analysis['has_seconds']}
- Contains milliseconds: {analysis['has_milliseconds']}
- Total time range: {analysis['time_range']}
- Sample values: {[str(x) for x in analysis['sample_values']]}"""

# If your datetime column is named 'datetime':
results = analyze_datetime_granularity_df(df)

# Or if your datetime column has a different name:
results = analyze_datetime_granularity_df(df, datetime_col='datetime')

# Print the formatted results
print(format_time_analysis(results))

In [ ]:
# prompt: group by longitude latitude for count of rows, order by count

import pandas as pd

# Assuming your DataFrame is named 'df'
grouped = df.groupby(['Longitude', 'Latitude']).size().reset_index(name='counts')
ordered_grouped = grouped.sort_values('counts', ascending=False)
ordered_grouped


^ This confirms that each longitude and latitude occurs only once.

In [ ]:
# prompt: # prompt: # prompt: group by date time get count
# # import pandas as pd
# # # Assuming your DataFrame is named 'df' and 'datetime' column exists
# # # and is already in datetime format as per your previous code.
# # # Group by date and get the count
# # df['date'] = df['datetime'].dt.date  # Extract the date part
# # date_counts = df.groupby('date').size().reset_index(name='count')
# # date_counts
# # group by date time
# # Group by datetime and get the count
# datetime_counts = df.groupby('datetime').size().reset_index(name='count')
# datetime_counts
# Get count, average Longitude, average Latitude, average UHI Index and median UHI Index

import pandas as pd

# Assuming your DataFrame is named 'df' and 'datetime' column exists
# and is already in datetime format.

# Group by datetime and get the count, along with other aggregated values
datetime_stats = df.groupby('datetime').agg(
    count=('UHI Index', 'count'),
    mean_longitude=('Longitude', 'mean'),
    mean_latitude=('Latitude', 'mean'),
    mean_uhi=('UHI Index', 'mean'),
    median_uhi=('UHI Index', 'median')
).reset_index()

datetime_stats


In [ ]:
# prompt: Distribution of UHI Index

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'uhi_index' is the column containing the UHI index values
plt.figure(figsize=(10, 6))
sns.histplot(df['UHI Index'], kde=True)  # kde=True adds a kernel density estimate
plt.title('Distribution of UHI Index')
plt.xlabel('UHI Index')
plt.ylabel('Frequency')
plt.show()


# Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(y=df['UHI Index'])
plt.title('Boxplot of UHI Index')
plt.ylabel('UHI Index')
plt.show()


In [ ]:
# prompt: examine outliers by iqr method

import pandas as pd

# Calculate Q1, Q3, and IQR
Q1 = df['UHI Index'].quantile(0.25)
Q3 = df['UHI Index'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[(df['UHI Index'] < lower_bound) | (df['UHI Index'] > upper_bound)]

# Print or display outliers
print("Outliers (IQR Method):")
outliers

# You can also visualize the outliers using a boxplot (already shown in previous code)
# or by creating a scatter plot highlighting outliers


In [ ]:
# prompt: remove outliers

# Remove outliers
df_no_outliers = df[(df['UHI Index'] >= lower_bound) & (df['UHI Index'] <= upper_bound)]

# Print or display the DataFrame without outliers
print("DataFrame without outliers:")
df_no_outliers


In [ ]:
# prompt: Distribution of UHI Index

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'uhi_index' is the column containing the UHI index values
plt.figure(figsize=(10, 6))
sns.histplot(df_no_outliers['UHI Index'], kde=True)  # kde=True adds a kernel density estimate
plt.title('Distribution of UHI Index')
plt.xlabel('UHI Index')
plt.ylabel('Frequency')
plt.show()


# Boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(y=df_no_outliers['UHI Index'])
plt.title('Boxplot of UHI Index')
plt.ylabel('UHI Index')
plt.show()


In [ ]:
# prompt: group Longitude Latitude aggregate average UHI Index

import pandas as pd

# Assuming 'df_no_outliers' is your DataFrame without outliers
# Group by 'Longitude' and 'Latitude' and calculate the average 'UHI Index'
grouped_df = df_no_outliers.groupby(['Longitude', 'Latitude'])['UHI Index'].mean().reset_index()

# Display the grouped DataFrame
display(grouped_df)


In [ ]:
df.shape[0] - grouped_df.shape[0]

In [ ]:
# prompt: plot distribution of UHI Index again

# Assuming 'df_no_outliers' is your DataFrame without outliers from the previous code

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'uhi_index' is the column containing the UHI index values
plt.figure(figsize=(10, 6))
sns.histplot(grouped_df['UHI Index'], kde=True)  # kde=True adds a kernel density estimate
plt.title('Distribution of UHI Index (Average)')
plt.xlabel('UHI Index')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# prompt: distribution of minute

import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'df' is your DataFrame and 'minute' column exists

plt.figure(figsize=(10, 6))
sns.histplot(df['minute'], kde=True, bins=60)  # Use bins=60 for minutes
plt.title('Distribution of Minute')
plt.xlabel('Minute')
plt.ylabel('Frequency')
plt.show()


# Model

In [ ]:
# prompt: cross fold validation of linear regression, ridge regression, lasso regression, elastic net, naive bayes regression, decision tree, random forest, light gbm, xgboost, svr, and knn regression and report the results sorted in descending order of R2 score.

import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

# Assuming 'grouped_df' is your preprocessed DataFrame
# Replace 'UHI Index' with the actual target variable column name if different
X = grouped_df.drop('UHI Index', axis=1)
y = grouped_df['UHI Index']

# Create a dictionary of models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Elastic Net': ElasticNet(),
    'Naive Bayes Regression': GaussianNB(),  # Note: Naive Bayes is a classification algorithm, use with caution for regression
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'LightGBM': LGBMRegressor(),
    'XGBoost': XGBRegressor(),
    'SVR': SVR(),
    'KNN Regression': KNeighborsRegressor()
}

# Perform cross-validation and store results
results = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42) # Example 5-fold CV

for name, model in models.items():
  try:
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
    results[name] = np.mean(cv_scores)
  except Exception as e:
    print(f"Error with {name}: {e}")
    results[name] = np.nan

# Sort results by R-squared score
sorted_results = dict(sorted(results.items(), key=lambda item: item[1], reverse=True))

# Print the sorted results
for name, score in sorted_results.items():
  print(f"{name}: {score:.4f}")


# Interpret

In [ ]:
# prompt: interpret the model for insights on the feature importance

import pandas as pd
import matplotlib.pyplot as plt
# Assuming 'X' is your feature matrix and 'models' is your dictionary of trained models.
# You'll need to adapt this based on the specific model you choose to interpret.

importances = {}

# Example: Get feature importances from a RandomForestRegressor
model = models['Random Forest']  # Or whichever model you want to analyze
if isinstance(model, RandomForestRegressor):
    model.fit(X,y)
    importances = model.feature_importances_
    feature_names = X.columns
    feature_importance_df = pd.DataFrame({'feature':feature_names, 'importance': importances})
    print(feature_importance_df.sort_values('importance', ascending=False))

# Example using permutation importance for any model:
from sklearn.inspection import permutation_importance
result = permutation_importance(model, X, y, n_repeats=10, random_state=42, n_jobs=2)
sorted_idx = result.importances_mean.argsort()

fig, ax = plt.subplots()
ax.boxplot(result.importances[sorted_idx].T, vert=False, labels=X.columns[sorted_idx])
ax.set_title("Permutation Importances (test set)")
fig.tight_layout()
plt.show()


#More detailed explanation of the feature importance scores based on the selected model.
# For tree-based models, higher values indicate stronger predictive power.
# For linear models, the coefficients show the effect of each feature on the target.
# If using permutation importance, the ranking reflects how much the model's accuracy decreases when the feature is randomly shuffled.


In [ ]:
import matplotlib.pyplot as plt
# Assuming 'grouped_df' is your DataFrame with 'Latitude' and 'UHI Index' columns
plt.figure(figsize=(10, 6))
sns.regplot(x='Latitude', y='UHI Index', data=grouped_df, scatter_kws={"color": "blue"}, line_kws={"color": "red"})
plt.xlabel('Latitude')
plt.ylabel('UHI Index')
plt.title('UHI Index vs. Latitude')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
# Assuming 'grouped_df' is your DataFrame with 'Latitude' and 'UHI Index' columns
plt.figure(figsize=(10, 6))
sns.regplot(x='Longitude', y='UHI Index', data=grouped_df, scatter_kws={"color": "blue"}, line_kws={"color": "red"})
plt.xlabel('Latitude')
plt.ylabel('UHI Index')
plt.title('UHI Index vs. Latitude')
plt.grid(True)
plt.show()


# Submission

In [ ]:
# prompt: retrain the best performing model on all the data (without cross-fold validation)

# Assuming 'X' and 'y' are your features and target variable from the grouped_df
# and 'sorted_results' contains the cross-validation results.

# Retrain the best performing model on the entire dataset
best_model_name = list(sorted_results.keys())[1] # Get the name of the best model
best_model = models[best_model_name]

best_model.fit(X, y) # Fit on the entire dataset

# Now 'best_model' is trained on all the data and ready for predictions on new data.
# You can save this model for later use if needed.
import joblib
joblib.dump(best_model, f'{best_model_name}_final_model.pkl')


In [ ]:
# prompt: use the model to predict X and use y as y_true and report the r2

from sklearn.metrics import r2_score

# Assuming 'best_model' is already trained and 'X' contains the features for prediction
# and 'y' is your true target values

y_pred = best_model.predict(X)
r2 = r2_score(y, y_pred)
print(f"R-squared score on the entire dataset: {r2}")


In [ ]:
# prompt: load submission.csv and use the model to predict the UHI_Index

import pandas as pd
# Load the submission data
submission_df = pd.read_csv("submission.csv")

submission_X = submission_df.drop('UHI Index', axis=1)

# Choose the best-performing model from cross-validation results
best_model_name = 'Random Forest'  # Replace with the actual best model
best_model = models[best_model_name]

# Retrain the best model on the entire training dataset (grouped_df)
best_model.fit(X, y)


# Make predictions on the submission data
submission_predictions = best_model.predict(submission_X)

# Create a new DataFrame for submission
submission_results = submission_X
submission_results['UHI_Index'] = submission_predictions

# Save the predictions to a CSV file
submission_results.to_csv('submission_predictions.csv', index=False)
